In [35]:
# Maybe we focus on subset of bts icaos???
# forward and backwards fill and filters to get out outliers of callsign. We can look at statistics of how far.
# See how much "noise" is in callsign field

In [36]:
from datetime import date
import polars as pl
from data_engineering.adsb.read_adsb import read_adsb
df = read_adsb(date(2026,3,1), icaos=["a19a21"])

In [37]:
COLUMNS = ["time", "icao", "callsign"]

In [38]:
from aircraft.callsign import is_valid_callsign
df = df.with_columns(is_valid_callsign().fill_null(False).alias("is_valid_callsign"))
df = df.with_columns(pl.when(pl.col("is_valid_callsign") == False).then(pl.lit(None)).otherwise(pl.col("callsign")).alias("callsign")).drop("is_valid_callsign")
df = df.sort(["icao", "time"]).with_columns(
    pl.col("callsign").forward_fill().backward_fill().over("icao")
)
assert len(df.filter(pl.col("callsign").is_null())) == 0
display(df.select(COLUMNS))

time,icao,callsign
datetime[ms],str,str
2026-03-01 00:00:00.580,"""a19a21""","""ASA826"""
2026-03-01 00:00:01.630,"""a19a21""","""ASA826"""
2026-03-01 00:00:02.480,"""a19a21""","""ASA826"""
2026-03-01 00:00:03.640,"""a19a21""","""ASA826"""
2026-03-01 00:00:04.740,"""a19a21""","""ASA826"""
…,…,…
2026-03-01 22:45:58.770,"""a19a21""","""ASA834"""
2026-03-01 22:46:01.120,"""a19a21""","""ASA834"""
2026-03-01 22:46:01.880,"""a19a21""","""ASA834"""


In [39]:
df = df.with_columns((pl.col("callsign") != pl.col("callsign").shift(1).fill_null(True)).cum_sum().alias("range_id"))
df = df.group_by("range_id", maintain_order=True).agg(
        pl.col("time").first().alias("start_time"),
        pl.col("time").last().alias("last_seen_time"),
        pl.col("callsign").first(),
    )
df

range_id,start_time,last_seen_time,callsign
u32,datetime[ms],datetime[ms],str
1,2026-03-01 00:00:00.580,2026-03-01 01:35:57.240,"""ASA826"""
2,2026-03-01 01:36:17.190,2026-03-01 18:35:48.140,"""ASA825"""
3,2026-03-01 18:36:20.750,2026-03-01 20:31:36.160,"""ASA1033"""
4,2026-03-01 20:31:42.010,2026-03-01 22:13:55.120,"""ASA1044"""
5,2026-03-01 22:14:16.520,2026-03-01 22:47:02.760,"""ASA834"""


## Callsign ranges and lookup

Collapse consecutive rows with the same callsign into half-open `[start_time, end_time)` intervals. A range ends when the next callsign is first observed; the final range ends one millisecond after the final message. This represents the inferred last-known callsign and therefore spans gaps between ADS-B messages.

In [ ]:
df_callsign_ranges = (
    df.select(COLUMNS)
    .drop_nulls("callsign")
    .sort(["icao", "time"])
    .with_columns(
        (pl.col("callsign") != pl.col("callsign").shift())
        .fill_null(True)
        .cum_sum()
        .over("icao")
        .alias("range_id")
    )
    .group_by(["icao", "range_id"], maintain_order=True)
    .agg(
        pl.col("time").first().alias("start_time"),
        pl.col("time").last().alias("last_seen_time"),
        pl.col("callsign").first(),
    )
    .with_columns(
        pl.col("start_time")
        .shift(-1)
        .over("icao")
        .fill_null(pl.col("last_seen_time") + pl.duration(milliseconds=1))
        .alias("end_time")
    )
    .select("icao", "start_time", "end_time", "callsign")
)

display(df_callsign_ranges)

icao,start_time,end_time,callsign
str,datetime[ms],datetime[ms],str
"""a19a21""",2026-03-01 00:00:00.580,2026-03-01 01:36:17.190,"""ASA826"""
"""a19a21""",2026-03-01 01:36:17.190,2026-03-01 18:36:20.750,"""ASA825"""
"""a19a21""",2026-03-01 18:36:20.750,2026-03-01 20:31:42.010,"""ASA1033"""
"""a19a21""",2026-03-01 20:31:42.010,2026-03-01 22:14:16.520,"""ASA1044"""
"""a19a21""",2026-03-01 22:14:16.520,2026-03-01 22:47:02.761,"""ASA834"""


In [ ]:
from bisect import bisect_right
from dataclasses import dataclass
from datetime import datetime


@dataclass(frozen=True)
class TimeRange:
    start_time: datetime
    end_time: datetime
    value: str


ranges_by_icao: dict[str, list[TimeRange]] = {}
for row in df_callsign_ranges.iter_rows(named=True):
    ranges_by_icao.setdefault(row["icao"], []).append(
        TimeRange(row["start_time"], row["end_time"], row["callsign"])
    )

range_starts_by_icao = {
    icao: [time_range.start_time for time_range in ranges]
    for icao, ranges in ranges_by_icao.items()
}


def lookup(icao: str, t: datetime) -> str | None:
    ranges = ranges_by_icao.get(icao)
    if not ranges:
        return None

    range_index = bisect_right(range_starts_by_icao[icao], t) - 1
    if range_index < 0:
        return None

    time_range = ranges[range_index]
    return time_range.value if t < time_range.end_time else None


lookup("a19a21", datetime(2026, 3, 1, 20, 0))